# Insensitive Flux Profile
Here we want to analyze the Fluxonium circuit (shown below). Again as in the c-shunted qubit notebook, we use the target_fluxqubit as target data and optimize the circuit parameters with Gradient Descent. For are more detailed explanation of the steps, check out c-shunted qubit example first.

<img src="fluxonium.jpg" alt="Circuit" width="300" height="300"/>

## Basics

In [63]:
from des_scq import models
from des_scq.circuit import Kerman, Charge
from des_scq.optimization import Optimization
from numpy import arange,linspace
from des_scq.utils import plotCompare
from torch import set_num_threads
set_num_threads(32)
import pickle
from des_scq.discovery import lossTransition
from torch import tensor, float32 as float, stack, float64 as double

The following workflow is similar to the one in the notebook "c-shunted qubit". We extract the target data for the energies $E_{10}$, $E_{21}$.

In [ ]:
with open('target_fluxqubit.pick', 'rb') as content: target_info = pickle.load(content)
target_spectrum = target_info['spectrum']

In [65]:
E10 = target_spectrum[:,1] - target_spectrum[:,0]
E21 = target_spectrum[:,2] - target_spectrum[:,1]

target = {'E10':E10[[0,10,20,30,-1]],'E21':E21[[0,10,20,30,-1]]}

Build circuit, with the function fluxonium() found in models.py. We want to work in Kerman representation.

In [66]:
rep = Kerman # choose Kerman basis
basis = {'O':[64],'J':[],'I':[]} # choose basis size
circuit = models.fluxonium(rep, basis,El=5.,Ec=5.,Ej=5.) 

In [67]:
nodeDistribution = circuit.kermanDistribution()
print('The number of oscillator modes N_o is',nodeDistribution[0])
print('The number of island modes N_i is',nodeDistribution[1])
print('The number of josephson modes N_j is',nodeDistribution[2])

The number of oscillator modes N_o is 1
The number of island modes N_i is 0
The number of josephson modes N_j is 0


As we can see there is only oscillator modes so the circuit complies with oscillator basis.

## Pre-Optimization Fluxonium
We want to plot the pre-optimization spectrum of the fluxonium qubit.

In [68]:
# set list of tensors with external flux values
flux_range = tensor(linspace(0,1,5,endpoint=True))
flux_profile = [[flux] for flux in flux_range]

In [69]:
# compute Hamiltonians
H_LC = circuit.hamiltonianLC()
H_J = circuit.hamiltonianJosephson

In [70]:
# compute Spectrum pre optimization
preoptim_Spectrum = circuit.spectrumManifold(flux_profile)

In [71]:
# extract eigenenergies from Spectrum and norm them to ground state
pre_Ex = [spectrum for spectrum,state in preoptim_Spectrum] 
pre_Ex = stack(pre_Ex).T 
pre_Ex = pre_Ex.detach().numpy()
pre_Ex = pre_Ex - pre_Ex[0] 

print('The transition energies E(1->0) for the different flux values in flux_range are', pre_Ex[1])
print('The transition energies E(2->1) for the different flux values in flux_range are', pre_Ex[2]-pre_Ex[1])
print('The transition energies E(2->1) for the different flux values in flux_range are', pre_Ex[3]-pre_Ex[3])
print('The transition energies E(2->0) for the different flux values in flux_range are', pre_Ex[2])

The transition energies E(1->0) for the different flux values in flux_range are [17.60698193 15.15053755 10.69345067 15.15053773 17.60698193]
The transition energies E(2->1) for the different flux values in flux_range are [15.43816104 13.74761412 13.45621048 13.74761419 15.43816104]
The transition energies E(2->1) for the different flux values in flux_range are [0. 0. 0. 0. 0.]
The transition energies E(2->0) for the different flux values in flux_range are [33.04514297 28.89815167 24.14966115 28.89815192 33.04514297]


When computing the spectrum, try for different basis sizes and check which is the minimum basis size where the eigenenergies do not change anymore! 

In [72]:
# visualize excitation profile pre-optimization
plotCompare(flux_range,{'E10':pre_Ex[1],'E21':pre_Ex[2]-pre_Ex[1],'anharmonicity':pre_Ex[2]-2*pre_Ex[1]},
            'Excitation Profile : pre-optimization','external flux','energy(GHz)',export='pdf',size=(600,800))

# Optimization

In [73]:
lossObjective = lossTransition(tensor(target['E10'],dtype=double),tensor(target['E21'],dtype=double)) # define loss function
optim = Optimization(circuit,flux_profile, lossObjective) # call GradientDescent optimizer
optim.initAlgo(lr=0.1) # set learning rate
print(optim.optimizer)
dLogs,dParams,dCircuit = optim.optimization(iterations=1000) #run optimization

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.1
    maximize: False
    weight_decay: 0
)


In [74]:
# plot loss vs. iterations
plotCompare(dLogs.index,dLogs[['loss']],'Optimization-Flux Sensitivity','iteration',export='pdf',size=(600,800))

In [75]:
# plot circuit parameters vs iterations
plotCompare(dCircuit.index,dCircuit,'Optimization-Circuit Parameters',"iteration","energy(GHz)",export='pdf',size=(600,800))

Analytically the solution is $JJ=0$. However, the optimization stagnates asymptotically and therefore gradient descent algorithm improper!

# Post-optimization Fluxonium

Show starting point for circuit parameters:

In [76]:
dCircuit.iloc[0]

Cap    5.0
JJ     5.0
I      5.0
Name: 0, dtype: float64

Show end point for circuit parameters after optimization:

In [77]:
dCircuit.iloc[-1]

Cap      0.139899
JJ     150.273509
I      167.677521
Name: 1000, dtype: float64

Now plot the post optimization profile. All the steps here are analogous to the steps in section "Pre-optimization profile"

In [78]:
flux_profile = [[flux] for flux in flux_range]
flux_point = ('I')

In [79]:
H_LC = circuit.hamiltonianLC()
H_J = circuit.hamiltonianJosephson

In [80]:
post_Spectrum = circuit.spectrumManifold(flux_profile)

In [81]:
post_Ex = [spectrum for spectrum,state in post_Spectrum]
post_Ex = stack(post_Ex).T
post_Ex -= post_Ex[0]
post_Ex = post_Ex.detach().numpy()

In [82]:
# visualize excitation profile post-optimization
plotCompare(flux_range,{'Ist':post_Ex[1],'IInd':post_Ex[2],'anharmonicity':post_Ex[2]-2*post_Ex[1]},
            'Excitation Profile : post-optimization','external flux','energy(GHz)',export='pdf',size=(600,800))

# Circuit Comparison

In [83]:
# compare energy excitation E (1->0) pre- and post-optimization
plot = {'post':post_Ex[1],'pre':pre_Ex[1],'Target':target['E10']}
plotCompare(flux_range,plot,
            'Excitation(E10) Profile','external flux','energy(GHz)',export='pdf',size=(600,800))

In [84]:
# compare energy excitation E (2->1) pre- and post-optimization
plot = {'post IInd':post_Ex[2],'pre IInd':pre_Ex[2],'Target21':target['E21']}
plotCompare(flux_range,plot,
            'Excitation(E21) Profile','external flux','energy(GHz)',export='pdf',size=(600,800))